Подготовим реалистичный Dataset для этапа PoC.

# 0. Инициализация

In [ ]:
!pip -q install -U polars datasets huggingface_hub

# 1. Настройки выборки

In [ ]:
import polars as pl
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

YEARS = [2022, 2023, 2024]

# Для PoC берём ограниченный региональный срез.
# 7700 — Москва, 7800 — Санкт-Петербург, 5000 — Московская область.
REGION_TAXCODES = ["7700", "7800", "5000"]

# C — обрабатывающие производства
# F — строительство
# G — торговля
# J — информация и связь
# M — профессиональная, научная и техническая деятельность
OKVED_SECTIONS = ["C", "F", "G", "J", "M"]

# Годы, по которым считаем динамику.
BASE_YEAR = 2023
TARGET_YEAR = 2024
REQUIRED_YEARS = [BASE_YEAR, TARGET_YEAR]

# Размеры корзин итоговой PoC-выборки.
SAMPLE_BUCKET_SIZES = {
    "normal": 30,
    "revenue_drop_gt_30": 20,
    "negative_profit": 20,
    "assets_drop_gt_25": 15,
    "data_quality_issue": 15,
}

RANDOM_SEED = 42

PATH_TEMPLATE = "hf://datasets/irlspbru/RFSD/RFSD/year={year}/*.parquet"

# 2. Колонки и словари данных

In [ ]:
META_COLUMNS = [
    "year",
    "inn",
    "ogrn",
    "region",
    "region_taxcode",
    "creation_date",
    "dissolution_date",
    "age",
    "eligible",
    "filed",
    "imputed",
    "outlier",
    "okved",
    "okved_section",
]

FINANCIAL_COLUMNS = [
    "line_2110",  # revenue / выручка
    "line_2400",  # net_profit / чистая прибыль
    "line_1600",  # assets / активы
    "line_1300",  # equity / капитал
    "line_1400",  # longterm_liab / долгосрочные обязательства
    "line_1500",  # shortterm_liab / краткосрочные обязательства
    "line_1520",  # payables / кредиторская задолженность
    "line_1250",  # cash / денежные средства
]

COLUMN_RENAME = {
    "line_2110": "revenue",
    "line_2400": "net_profit",
    "line_1600": "assets",
    "line_1300": "equity",
    "line_1400": "longterm_liab",
    "line_1500": "shortterm_liab",
    "line_1520": "payables",
    "line_1250": "cash",
}

## 2.1. Производные группы колонок

In [ ]:
ID_COLUMNS = [
    "inn",
    "ogrn",
    "region",
    "region_taxcode",
    "okved",
    "okved_section",
]

DATE_COLUMNS = [
    "creation_date",
    "dissolution_date",
]

FLAG_COLUMNS = [
    "eligible",
    "filed",
    "imputed",
    "outlier",
]

RENAMED_FINANCIAL_COLUMNS = [
    "revenue",
    "net_profit",
    "assets",
    "equity",
    "longterm_liab",
    "shortterm_liab",
    "payables",
    "cash",
]

REPORT_COLUMNS = [
    "inn",
    "ogrn",
    "year",
    "region",
    "region_taxcode",
    "okved",
    "okved_section",
    "creation_date",
    "dissolution_date",
    "age",
    "eligible",
    "filed",
    "imputed",
    "outlier",
    "revenue",
    "net_profit",
    "assets",
    "equity",
    "longterm_liab",
    "shortterm_liab",
    "payables",
    "cash",
]

COMPANY_COLUMNS = [
    "inn",
    "ogrn",
    "region",
    "region_taxcode",
    "okved",
    "okved_section",
    "creation_date",
    "dissolution_date",
    "age",
]

QUALITY_FLAG_COLUMNS = [
    "filed",
    "imputed",
    "outlier",
]

RISK_COLUMNS = [
    "risk_revenue_drop_gt_30",
    "risk_negative_profit",
    "risk_assets_drop_gt_25",
    "risk_negative_equity",
    "risk_data_quality_issue",
]

DERIVED_FEATURE_COLUMNS = [
    f"revenue_{BASE_YEAR}",
    f"revenue_{TARGET_YEAR}",
    f"revenue_drop_{TARGET_YEAR}_pct",

    f"net_profit_{BASE_YEAR}",
    f"net_profit_{TARGET_YEAR}",

    f"assets_{BASE_YEAR}",
    f"assets_{TARGET_YEAR}",
    f"assets_drop_{TARGET_YEAR}_pct",

    f"equity_{BASE_YEAR}",
    f"equity_{TARGET_YEAR}",

    f"filed_{BASE_YEAR}",
    f"filed_{TARGET_YEAR}",

    f"imputed_{BASE_YEAR}",
    f"imputed_{TARGET_YEAR}",

    f"outlier_{BASE_YEAR}",
    f"outlier_{TARGET_YEAR}",
]

# 3. Загрузка RFSD
## 3.1. Авторизация

In [ ]:
import os
import getpass
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF_TOKEN: ")

login(token=HF_TOKEN, add_to_git_credential=False)

HF_TOKEN: ··········


## 3.2. Непосредственно загрузка

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

REPO_ID = "irlspbru/RFSD"
REPO_TYPE = "dataset"

LOCAL_RFSD_DIR = Path("data/rfsd_raw")
LOCAL_RFSD_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2022, 2023, 2024]

def download_rfsd_year(year: int) -> str:
    filename = f"RFSD/year={year}/part-0.parquet"

    local_path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        filename=filename,
        token=HF_TOKEN,
        local_dir=LOCAL_RFSD_DIR,
    )

    return local_path

YEAR_PATHS = {}

for year in YEARS:
    print(f"Downloading {year}...")
    YEAR_PATHS[year] = download_rfsd_year(year)
    print(YEAR_PATHS[year])

data/rfsd_raw/RFSD/year=2022/part-0.parquet
data/rfsd_raw/RFSD/year=2023/part-0.parquet
data/rfsd_raw/RFSD/year=2024/part-0.parquet


# 4. Чтение RFSD в рабочий DataFrame
## 4.1. Функция чтения одного года


In [ ]:
import polars as pl


def read_rfsd_year_local(year: int) -> pl.LazyFrame:
    path = YEAR_PATHS[year]

    lf = pl.scan_parquet(path)
    schema_names = lf.collect_schema().names()

    # year не читаем из parquet, потому что он задан партицией year=2022/year=2023/year=2024.
    wanted_columns = [
        c for c in (META_COLUMNS + FINANCIAL_COLUMNS)
        if c != "year"
    ]

    existing_columns = [c for c in wanted_columns if c in schema_names]
    missing_columns = [c for c in wanted_columns if c not in schema_names]

    if missing_columns:
        print(f"[WARN] year={year}: missing columns: {missing_columns}")

    lf = lf.select(existing_columns)

    # Добавляем год вручную.
    lf = lf.with_columns(
        pl.lit(year).cast(pl.Int32).alias("year")
    )

    # Переименовываем бухгалтерские строки в человекочитаемые названия.
    rename_map = {
        old: new
        for old, new in COLUMN_RENAME.items()
        if old in existing_columns
    }

    lf = lf.rename(rename_map)

    current_cols = lf.collect_schema().names()

    # Строковые поля.
    string_cols = [
        c for c in ID_COLUMNS
        if c in current_cols
    ]

    lf = lf.with_columns([
        pl.col(c)
        .cast(pl.Utf8, strict=False)
        .str.strip_chars()
        .alias(c)
        for c in string_cols
    ])

    # Флаги качества и применимости.
    flag_cols = [
        c for c in FLAG_COLUMNS
        if c in current_cols
    ]

    lf = lf.with_columns([
        pl.col(c).cast(pl.Float64, strict=False).alias(c)
        for c in flag_cols
    ])

    # Финансовые показатели.
    financial_cols = [
        c for c in RENAMED_FINANCIAL_COLUMNS
        if c in lf.collect_schema().names()
    ]

    lf = lf.with_columns([
        pl.col(c).cast(pl.Float64, strict=False).alias(c)
        for c in financial_cols
    ])

    # Базовые фильтры по идентификаторам.
    if "inn" in current_cols:
        lf = lf.filter(pl.col("inn").is_not_null())

    if "ogrn" in current_cols:
        lf = lf.filter(pl.col("ogrn").is_not_null())

    # Ограничиваем региональный срез.
    if "region_taxcode" in current_cols and REGION_TAXCODES:
        lf = lf.filter(pl.col("region_taxcode").is_in(REGION_TAXCODES))

    # Ограничиваем отраслевой срез.
    if "okved_section" in current_cols and OKVED_SECTIONS:
        lf = lf.filter(pl.col("okved_section").is_in(OKVED_SECTIONS))

    # ВАЖНО:
    # Пока НЕ фильтруем по eligible, filed, imputed, outlier.
    # Эти поля нужны дальше для формирования чистой выборки и корзины проблем качества данных.

    return lf

## 4.2. Собираем данные за все годы

In [ ]:
frames0 = [
    read_rfsd_year_local(year)
    for year in YEARS
]

raw0 = (
    pl.concat(frames0, how="diagonal_relaxed")
    .collect()
)

print(raw0.shape)

raw0.select("year").unique().sort("year")

(2114289, 22)


year
i32
2022
2023
2024


## 4.3. Быстрая проверка распределения по годам

In [ ]:
raw0.group_by("year").agg(
    pl.len().alias("rows")
).sort("year")

year,rows
i32,u32
2022,697288
2023,710546
2024,706455


## 4.4. Проверка регионов и секций ОКВЭД

In [ ]:
raw0.group_by("region_taxcode").agg(
    pl.len().alias("rows")
).sort("rows", descending=True)

region_taxcode,rows
str,u32
"""7700""",1334769
"""7800""",438830
"""5000""",340690


In [ ]:
raw0.group_by("okved_section").agg(
    pl.len().alias("rows")
).sort("rows", descending=True)

okved_section,rows
str,u32
"""G""",938459
"""F""",440511
"""M""",341998
"""C""",226807
"""J""",166514


## 4.5. Проверка служебных флагов

In [ ]:
for col in ["eligible", "filed", "imputed", "outlier"]:
    if col in raw0.columns:
        print(f"\n{col}")
        display(
            raw0.group_by(col).agg(
                pl.len().alias("rows")
            ).sort(col)
        )


eligible


eligible,rows
f64,u32
0.0,29733
1.0,2084556



filed


filed,rows
f64,u32
0.0,776027
1.0,1338262



imputed


imputed,rows
f64,u32
0.0,2078861
1.0,35428



outlier


outlier,rows
f64,u32
0.0,2114234
1.0,55


# 5. Формируем таблицу отчётности financial_reports

Теперь из raw0 сделаем нормальную длинную таблицу: одна строка = одна компания за один год.

In [ ]:
existing_report_cols = [
    c for c in REPORT_COLUMNS
    if c in raw0.columns
]

financial_reports = (
    raw0
    .select(existing_report_cols)
    .unique(subset=["inn", "year"], keep="first")
    .sort(["inn", "year"])
)

print(financial_reports.shape)

financial_reports.head()



(2114289, 22)


inn,ogrn,year,region,region_taxcode,okved,okved_section,creation_date,dissolution_date,age,eligible,filed,imputed,outlier,revenue,net_profit,assets,equity,longterm_liab,shortterm_liab,payables,cash
str,str,i32,str,str,str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""0100008846""","""1240100001679""",2024,"""moscow city""","""7700""","""71.12.45""","""M""",2024-06-10,null,0.0,1.0,1.0,0.0,0.0,3214.0,-796.0,7268.0,-786.0,7368.0,686.0,686.0,382.0
"""0105011190""","""1020100699508""",2022,"""moscow city""","""7700""","""69.20.10""","""M""",1994-02-02,null,28.0,1.0,1.0,0.0,0.0,4945.0,1035.0,2897.0,240.0,null,2657.0,2284.0,44.0
"""0105011190""","""1020100699508""",2023,"""moscow city""","""7700""","""69.20.10""","""M""",1994-02-02,null,29.0,1.0,1.0,0.0,0.0,4705.0,70.0,2335.0,310.0,null,2025.0,1557.0,106.0
"""0105011190""","""1020100699508""",2024,"""moscow city""","""7700""","""69.20.10""","""M""",1994-02-02,null,30.0,1.0,1.0,0.0,0.0,4167.0,345.0,1960.0,654.0,null,1306.0,742.0,26.0
"""0105032257""","""1020100701411""",2022,"""moscow reg.""","""5000""","""46.73""","""G""",2000-09-05,null,22.0,1.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null


In [ ]:
financial_reports.group_by("year").agg(
    pl.len().alias("companies_count")
).sort("year")

year,companies_count
i32,u32
2022,697288
2023,710546
2024,706455


# 6. Оставляем компании, у которых есть годы для сравнения

In [ ]:
companies_with_required_years = (
    financial_reports
    .filter(pl.col("year").is_in(REQUIRED_YEARS))
    .group_by("inn")
    .agg(
        pl.col("year").n_unique().alias("years_count")
    )
    .filter(pl.col("years_count") == len(REQUIRED_YEARS))
    .select("inn")
)

reports_2y = financial_reports.join(
    companies_with_required_years,
    on="inn",
    how="inner"
)

print(reports_2y.shape)

reports_2y.select("year").unique().sort("year")

reports_2y.select("inn").n_unique()

(1769317, 22)


621833

# 7. Справочник компаний и широкая витрина признаков
## 7.1. Справочник компаний

In [ ]:
company_dim = (
    reports_2y
    .sort(["inn", "year"])
    .group_by("inn")
    .agg([
        pl.col("ogrn").drop_nulls().last().alias("ogrn"),
        pl.col("region").drop_nulls().last().alias("region"),
        pl.col("region_taxcode").drop_nulls().last().alias("region_taxcode"),
        pl.col("okved").drop_nulls().last().alias("okved"),
        pl.col("okved_section").drop_nulls().last().alias("okved_section"),
        pl.col("creation_date").drop_nulls().last().alias("creation_date"),
        pl.col("dissolution_date").drop_nulls().last().alias("dissolution_date"),
        pl.col("age").drop_nulls().last().alias("age"),
    ])
    .with_columns(
        ("Компания ИНН " + pl.col("inn")).alias("company_label")
    )
)

print(company_dim.shape)
company_dim.head()

(621833, 10)


inn,ogrn,region,region_taxcode,okved,okved_section,creation_date,dissolution_date,age,company_label
str,str,str,str,str,str,date,date,f64,str
"""0105011190""","""1020100699508""","""moscow city""","""7700""","""69.20.10""","""M""",1994-02-02,null,30.0,"""Компания ИНН 0105011190"""
"""0105032257""","""1020100701411""","""moscow reg.""","""5000""","""46.73""","""G""",2000-09-05,null,24.0,"""Компания ИНН 0105032257"""
"""0105039252""","""1030100530162""","""moscow city""","""7700""","""71.10""","""M""",2003-03-24,null,21.0,"""Компания ИНН 0105039252"""
"""0105068870""","""1130105001135""","""moscow city""","""7700""","""46.71""","""G""",2013-04-02,null,11.0,"""Компания ИНН 0105068870"""
"""0105075839""","""1150105002300""","""moscow city""","""7700""","""46.90""","""G""",2015-11-03,null,9.0,"""Компания ИНН 0105075839"""


## 7.2. Широкая витрина по годам



In [ ]:
def first_value_for_year(col: str, year: int) -> pl.Expr:
    return (
        pl.col(col)
        .filter(pl.col("year") == year)
        .drop_nulls()
        .first()
        .alias(f"{col}_{year}")
    )


wide_exprs = []

for year in REQUIRED_YEARS:
    for col in RENAMED_FINANCIAL_COLUMNS + FLAG_COLUMNS:
        if col in reports_2y.columns:
            wide_exprs.append(first_value_for_year(col, year))

wide = (
    reports_2y
    .group_by("inn")
    .agg(wide_exprs)
)

print(wide.shape)
wide.head()

(621833, 25)


inn,revenue_2023,net_profit_2023,assets_2023,equity_2023,longterm_liab_2023,shortterm_liab_2023,payables_2023,cash_2023,eligible_2023,filed_2023,imputed_2023,outlier_2023,revenue_2024,net_profit_2024,assets_2024,equity_2024,longterm_liab_2024,shortterm_liab_2024,payables_2024,cash_2024,eligible_2024,filed_2024,imputed_2024,outlier_2024
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""0105011190""",4705.0,70.0,2335.0,310.0,null,2025.0,1557.0,106.0,1.0,1.0,0.0,0.0,4167.0,345.0,1960.0,654.0,null,1306.0,742.0,26.0,1.0,1.0,0.0,0.0
"""0105032257""",null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0
"""0105039252""",139826.0,6882.0,84945.0,43747.0,24634.0,16564.0,16528.0,0.0,1.0,1.0,0.0,0.0,null,null,100.0,100.0,null,null,null,100.0,1.0,1.0,0.0,0.0
"""0105068870""",177898.0,106.0,26397.0,393.0,null,26004.0,26004.0,null,1.0,1.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0
"""0105075839""",null,null,27554.0,36.0,null,27518.0,27518.0,null,1.0,1.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0


## 7.3. Объединяем справочник и витрину

In [ ]:
features = (
    company_dim
    .join(wide, on="inn", how="inner")
)

print(features.shape)
features.head()

(621833, 34)


inn,ogrn,region,region_taxcode,okved,okved_section,creation_date,dissolution_date,age,company_label,revenue_2023,net_profit_2023,assets_2023,equity_2023,longterm_liab_2023,shortterm_liab_2023,payables_2023,cash_2023,eligible_2023,filed_2023,imputed_2023,outlier_2023,revenue_2024,net_profit_2024,assets_2024,equity_2024,longterm_liab_2024,shortterm_liab_2024,payables_2024,cash_2024,eligible_2024,filed_2024,imputed_2024,outlier_2024
str,str,str,str,str,str,date,date,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""0105011190""","""1020100699508""","""moscow city""","""7700""","""69.20.10""","""M""",1994-02-02,null,30.0,"""Компания ИНН 0105011190""",4705.0,70.0,2335.0,310.0,null,2025.0,1557.0,106.0,1.0,1.0,0.0,0.0,4167.0,345.0,1960.0,654.0,null,1306.0,742.0,26.0,1.0,1.0,0.0,0.0
"""0105032257""","""1020100701411""","""moscow reg.""","""5000""","""46.73""","""G""",2000-09-05,null,24.0,"""Компания ИНН 0105032257""",null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0
"""0105039252""","""1030100530162""","""moscow city""","""7700""","""71.10""","""M""",2003-03-24,null,21.0,"""Компания ИНН 0105039252""",139826.0,6882.0,84945.0,43747.0,24634.0,16564.0,16528.0,0.0,1.0,1.0,0.0,0.0,null,null,100.0,100.0,null,null,null,100.0,1.0,1.0,0.0,0.0
"""0105068870""","""1130105001135""","""moscow city""","""7700""","""46.71""","""G""",2013-04-02,null,11.0,"""Компания ИНН 0105068870""",177898.0,106.0,26397.0,393.0,null,26004.0,26004.0,null,1.0,1.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0
"""0105075839""","""1150105002300""","""moscow city""","""7700""","""46.90""","""G""",2015-11-03,null,9.0,"""Компания ИНН 0105075839""",null,null,27554.0,36.0,null,27518.0,27518.0,null,1.0,1.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0


# 8. Расчёт риск-признаков
## 8.1. Вспомогательная функция

In [ ]:
def col_exists(df: pl.DataFrame, col: str) -> bool:
    return col in df.columns

## 8.2. Расчёт динамики

In [ ]:
exprs = []

revenue_base_col = f"revenue_{BASE_YEAR}"
revenue_target_col = f"revenue_{TARGET_YEAR}"

assets_base_col = f"assets_{BASE_YEAR}"
assets_target_col = f"assets_{TARGET_YEAR}"

if col_exists(features, revenue_base_col) and col_exists(features, revenue_target_col):
    exprs.append(
        pl.when(pl.col(revenue_base_col) > 0)
        .then(
            (pl.col(revenue_base_col) - pl.col(revenue_target_col))
            / pl.col(revenue_base_col)
            * 100
        )
        .otherwise(None)
        .alias(f"revenue_drop_{TARGET_YEAR}_pct")
    )

if col_exists(features, assets_base_col) and col_exists(features, assets_target_col):
    exprs.append(
        pl.when(pl.col(assets_base_col) > 0)
        .then(
            (pl.col(assets_base_col) - pl.col(assets_target_col))
            / pl.col(assets_base_col)
            * 100
        )
        .otherwise(None)
        .alias(f"assets_drop_{TARGET_YEAR}_pct")
    )

features = features.with_columns(exprs)

features.select([
    "inn",
    revenue_base_col,
    revenue_target_col,
    f"revenue_drop_{TARGET_YEAR}_pct",
    assets_base_col,
    assets_target_col,
    f"assets_drop_{TARGET_YEAR}_pct",
]).head(10)

inn,revenue_2023,revenue_2024,revenue_drop_2024_pct,assets_2023,assets_2024,assets_drop_2024_pct
str,f64,f64,f64,f64,f64,f64
"""0105011190""",4705.0,4167.0,11.434644,2335.0,1960.0,16.059957
"""0105032257""",null,null,null,null,null,null
"""0105039252""",139826.0,null,null,84945.0,100.0,99.882277
"""0105068870""",177898.0,null,null,26397.0,null,null
"""0105075839""",null,null,null,27554.0,null,null
"""0107014165""",null,null,null,null,null,null
"""0107022409""",619043.0,null,null,149043.0,null,null
"""0107028400""",null,null,null,null,null,null
"""0202008852""",null,null,null,6275.0,6319.0,-0.701195


## 8.3. Риск-флаги

In [ ]:
risk_exprs = []

revenue_drop_col = f"revenue_drop_{TARGET_YEAR}_pct"
assets_drop_col = f"assets_drop_{TARGET_YEAR}_pct"
net_profit_target_col = f"net_profit_{TARGET_YEAR}"
equity_target_col = f"equity_{TARGET_YEAR}"

if col_exists(features, revenue_drop_col):
    risk_exprs.append(
        (pl.col(revenue_drop_col) >= 30)
        .fill_null(False)
        .alias("risk_revenue_drop_gt_30")
    )

if col_exists(features, net_profit_target_col):
    risk_exprs.append(
        (pl.col(net_profit_target_col) < 0)
        .fill_null(False)
        .alias("risk_negative_profit")
    )

if col_exists(features, assets_drop_col):
    risk_exprs.append(
        (pl.col(assets_drop_col) >= 25)
        .fill_null(False)
        .alias("risk_assets_drop_gt_25")
    )

if col_exists(features, equity_target_col):
    risk_exprs.append(
        (pl.col(equity_target_col) < 0)
        .fill_null(False)
        .alias("risk_negative_equity")
    )

features = features.with_columns(risk_exprs)

features.head()

inn,ogrn,region,region_taxcode,okved,okved_section,creation_date,dissolution_date,age,company_label,revenue_2023,net_profit_2023,assets_2023,equity_2023,longterm_liab_2023,shortterm_liab_2023,payables_2023,cash_2023,eligible_2023,filed_2023,imputed_2023,outlier_2023,revenue_2024,net_profit_2024,assets_2024,equity_2024,longterm_liab_2024,shortterm_liab_2024,payables_2024,cash_2024,eligible_2024,filed_2024,imputed_2024,outlier_2024,revenue_drop_2024_pct,assets_drop_2024_pct,risk_revenue_drop_gt_30,risk_negative_profit,risk_assets_drop_gt_25,risk_negative_equity
str,str,str,str,str,str,date,date,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool
"""0105011190""","""1020100699508""","""moscow city""","""7700""","""69.20.10""","""M""",1994-02-02,null,30.0,"""Компания ИНН 0105011190""",4705.0,70.0,2335.0,310.0,null,2025.0,1557.0,106.0,1.0,1.0,0.0,0.0,4167.0,345.0,1960.0,654.0,null,1306.0,742.0,26.0,1.0,1.0,0.0,0.0,11.434644,16.059957,false,false,false,false
"""0105032257""","""1020100701411""","""moscow reg.""","""5000""","""46.73""","""G""",2000-09-05,null,24.0,"""Компания ИНН 0105032257""",null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0,null,null,false,false,false,false
"""0105039252""","""1030100530162""","""moscow city""","""7700""","""71.10""","""M""",2003-03-24,null,21.0,"""Компания ИНН 0105039252""",139826.0,6882.0,84945.0,43747.0,24634.0,16564.0,16528.0,0.0,1.0,1.0,0.0,0.0,null,null,100.0,100.0,null,null,null,100.0,1.0,1.0,0.0,0.0,null,99.882277,false,false,true,false
"""0105068870""","""1130105001135""","""moscow city""","""7700""","""46.71""","""G""",2013-04-02,null,11.0,"""Компания ИНН 0105068870""",177898.0,106.0,26397.0,393.0,null,26004.0,26004.0,null,1.0,1.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0,null,null,false,false,false,false
"""0105075839""","""1150105002300""","""moscow city""","""7700""","""46.90""","""G""",2015-11-03,null,9.0,"""Компания ИНН 0105075839""",null,null,27554.0,36.0,null,27518.0,27518.0,null,1.0,1.0,0.0,0.0,null,null,null,null,null,null,null,null,1.0,0.0,0.0,0.0,null,null,false,false,false,false


## 8.4. Признак проблем качества данных

In [ ]:
data_quality_conditions = []

filed_target_col = f"filed_{TARGET_YEAR}"
imputed_target_col = f"imputed_{TARGET_YEAR}"
outlier_target_col = f"outlier_{TARGET_YEAR}"

if col_exists(features, filed_target_col):
    data_quality_conditions.append(
        (pl.col(filed_target_col) == 0.0).fill_null(False)
    )

if col_exists(features, imputed_target_col):
    data_quality_conditions.append(
        (pl.col(imputed_target_col) == 1.0).fill_null(False)
    )

if col_exists(features, outlier_target_col):
    data_quality_conditions.append(
        (pl.col(outlier_target_col) == 1.0).fill_null(False)
    )

if data_quality_conditions:
    data_quality_expr = data_quality_conditions[0]

    for condition in data_quality_conditions[1:]:
        data_quality_expr = data_quality_expr | condition

    features = features.with_columns(
        data_quality_expr.alias("risk_data_quality_issue")
    )
else:
    features = features.with_columns(
        pl.lit(False).alias("risk_data_quality_issue")
    )

# 9. Диагностика доступных корзин

## 9.1. Смотрим, сколько компаний получилось по каждому признаку.

In [ ]:
risk_cols = [
    c for c in [
        "risk_revenue_drop_gt_30",
        "risk_negative_profit",
        "risk_assets_drop_gt_25",
        "risk_negative_equity",
        "risk_data_quality_issue",
    ]
    if c in features.columns
]

features.select([
    pl.len().alias("total_companies"),
    *[
        pl.col(c).sum().alias(c)
        for c in risk_cols
    ]
])

total_companies,risk_revenue_drop_gt_30,risk_negative_profit,risk_assets_drop_gt_25,risk_negative_equity,risk_data_quality_issue
u32,u32,u32,u32,u32,u32
621833,76992,86331,65456,64186,217028


## 9.2. Формируем PoC-выборку по корзинам

Опишем служебные функции

In [ ]:
def safe_sample(df: pl.DataFrame, n: int, seed: int) -> pl.DataFrame:
    """
    Безопасная выборка: если строк меньше, чем нужно, берём сколько есть.
    """
    if df.height == 0:
        return df

    return df.sample(
        n=min(n, df.height),
        seed=seed,
        shuffle=True,
    )

selected_inns = set()


def exclude_already_selected(df: pl.DataFrame) -> pl.DataFrame:
    """
    Исключаем компании, которые уже попали в предыдущие корзины.
    Это нужно, чтобы одна компания не дублировалась в нескольких bucket'ах.
    """
    if not selected_inns:
        return df

    return df.filter(~pl.col("inn").is_in(list(selected_inns)))


### 9.2.1. Корзина normal

Нормальная компания для PoC — это компания без наших базовых риск-флагов, с положительной выручкой и положительной прибылью.

In [ ]:
normal_conditions = []

if f"revenue_{TARGET_YEAR}" in features.columns:
    normal_conditions.append(pl.col(f"revenue_{TARGET_YEAR}") > 0)

if f"net_profit_{TARGET_YEAR}" in features.columns:
    normal_conditions.append(pl.col(f"net_profit_{TARGET_YEAR}") > 0)

for risk_col in [
    "risk_revenue_drop_gt_30",
    "risk_negative_profit",
    "risk_assets_drop_gt_25",
    "risk_negative_equity",
    "risk_data_quality_issue",
]:
    if risk_col in features.columns:
        normal_conditions.append(pl.col(risk_col) == False)

normal_filter = normal_conditions[0]

for condition in normal_conditions[1:]:
    normal_filter = normal_filter & condition

normal = (
    safe_sample(
        features.filter(normal_filter),
        n=SAMPLE_BUCKET_SIZES.get("normal", 30),
        seed=RANDOM_SEED,
    )
    .with_columns(
        pl.lit("normal").alias("sample_bucket")
    )
)

selected_inns.update(normal["inn"].to_list())

print(normal.shape)

(30, 42)


### 9.2.2. Корзина revenue_drop_gt_30

In [ ]:
if "risk_revenue_drop_gt_30" in features.columns:
    revenue_drop = (
        safe_sample(
            exclude_already_selected(features)
            .filter(pl.col("risk_revenue_drop_gt_30") == True),
            n=SAMPLE_BUCKET_SIZES.get("revenue_drop_gt_30", 20),
            seed=RANDOM_SEED + 1,
        )
        .with_columns(
            pl.lit("revenue_drop_gt_30").alias("sample_bucket")
        )
    )
else:
    revenue_drop = features.head(0).with_columns(
        pl.lit("revenue_drop_gt_30").alias("sample_bucket")
    )

selected_inns.update(revenue_drop["inn"].to_list())

print(revenue_drop.shape)


(20, 42)


### 9.2.3. Корзина negative_profit

In [ ]:
if "risk_negative_profit" in features.columns:
    negative_profit = (
        safe_sample(
            exclude_already_selected(features)
            .filter(pl.col("risk_negative_profit") == True),
            n=SAMPLE_BUCKET_SIZES.get("negative_profit", 20),
            seed=RANDOM_SEED + 2,
        )
        .with_columns(
            pl.lit("negative_profit").alias("sample_bucket")
        )
    )
else:
    negative_profit = features.head(0).with_columns(
        pl.lit("negative_profit").alias("sample_bucket")
    )

selected_inns.update(negative_profit["inn"].to_list())

print(negative_profit.shape)


(20, 42)


### 9.2.4. Корзина assets_drop_gt_25

In [ ]:
if "risk_assets_drop_gt_25" in features.columns:
    assets_drop = (
        safe_sample(
            exclude_already_selected(features)
            .filter(pl.col("risk_assets_drop_gt_25") == True),
            n=SAMPLE_BUCKET_SIZES.get("assets_drop_gt_25", 15),
            seed=RANDOM_SEED + 3,
        )
        .with_columns(
            pl.lit("assets_drop_gt_25").alias("sample_bucket")
        )
    )
else:
    assets_drop = features.head(0).with_columns(
        pl.lit("assets_drop_gt_25").alias("sample_bucket")
    )

selected_inns.update(assets_drop["inn"].to_list())

print(assets_drop.shape)

(15, 42)


### 9.4.5. Корзина data_quality_issue

In [ ]:
if "risk_data_quality_issue" in features.columns:
    data_quality = (
        safe_sample(
            exclude_already_selected(features)
            .filter(pl.col("risk_data_quality_issue") == True),
            n=SAMPLE_BUCKET_SIZES.get("data_quality_issue", 15),
            seed=RANDOM_SEED + 4,
        )
        .with_columns(
            pl.lit("data_quality_issue").alias("sample_bucket")
        )
    )
else:
    data_quality = features.head(0).with_columns(
        pl.lit("data_quality_issue").alias("sample_bucket")
    )

selected_inns.update(data_quality["inn"].to_list())

print(data_quality.shape)

(15, 42)


## 9.5. Собираем итоговую выборку

In [ ]:
sample_companies = (
    pl.concat(
        [
            normal,
            revenue_drop,
            negative_profit,
            assets_drop,
            data_quality,
        ],
        how="diagonal_relaxed",
    )
    .unique(subset=["inn"], keep="first")
)

print(sample_companies.shape)

sample_companies.group_by("sample_bucket").agg(
    pl.len().alias("companies_count")
).sort("sample_bucket")

(100, 42)


sample_bucket,companies_count
str,u32
"""assets_drop_gt_25""",15
"""data_quality_issue""",15
"""negative_profit""",20
"""normal""",30
"""revenue_drop_gt_30""",20


## 9.6. Проверяем итоговую выборку визуально

In [ ]:
preview_cols = [
    "inn",
    "ogrn",
    "region",
    "region_taxcode",
    "okved_section",
    "company_label",
    f"revenue_{BASE_YEAR}",
    f"revenue_{TARGET_YEAR}",
    f"revenue_drop_{TARGET_YEAR}_pct",
    f"net_profit_{TARGET_YEAR}",
    f"assets_{BASE_YEAR}",
    f"assets_{TARGET_YEAR}",
    f"assets_drop_{TARGET_YEAR}_pct",
    f"equity_{TARGET_YEAR}",
    "risk_revenue_drop_gt_30",
    "risk_negative_profit",
    "risk_assets_drop_gt_25",
    "risk_negative_equity",
    "risk_data_quality_issue",
]


sample_preview_cols = [
    "sample_bucket",
    *preview_cols,
]

sample_preview_cols = [
    c for c in sample_preview_cols
    if c in sample_companies.columns
]

sample_companies.select(sample_preview_cols).sort([
    "sample_bucket",
    "inn",
]).head()

sample_bucket,inn,ogrn,region,region_taxcode,okved_section,company_label,revenue_2023,revenue_2024,revenue_drop_2024_pct,net_profit_2024,assets_2023,assets_2024,assets_drop_2024_pct,equity_2024,risk_revenue_drop_gt_30,risk_negative_profit,risk_assets_drop_gt_25,risk_negative_equity,risk_data_quality_issue
str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool
"""assets_drop_gt_25""","""5040164597""","""1195027023850""","""moscow reg.""","""5000""","""G""","""Компания ИНН 5040164597""",16233.0,87572.0,-439.468983,8666.0,53680.0,36989.0,31.093517,15666.0,false,false,true,false,false
"""assets_drop_gt_25""","""5075017709""","""1045011650089""","""moscow reg.""","""5000""","""C""","""Компания ИНН 5075017709""",524.0,135.0,74.236641,3652.0,34404.0,22650.0,34.164632,4841.0,true,false,true,false,false
"""assets_drop_gt_25""","""7701726555""","""5077746878920""","""moscow city""","""7700""","""F""","""Компания ИНН 7701726555""",null,null,null,-19665.0,74282.0,53884.0,27.460219,53262.0,false,true,true,false,false
"""assets_drop_gt_25""","""7705100222""","""1027700157000""","""moscow city""","""7700""","""G""","""Компания ИНН 7705100222""",4923.0,4475.0,9.100142,16.0,1238.0,880.0,28.917609,169.0,false,false,true,false,false
"""assets_drop_gt_25""","""7706659743""","""5077746859031""","""moscow city""","""7700""","""G""","""Компания ИНН 7706659743""",808.0,1313.0,-62.5,31.0,1345.0,250.0,81.412639,190.0,false,false,true,false,false


## 9.7. Проверяем, что выборка пригодна для PoC

In [ ]:
sample_companies.select([
    pl.len().alias("total_companies"),
    pl.col("inn").n_unique().alias("unique_inn"),
    pl.col("ogrn").n_unique().alias("unique_ogrn"),
])

total_companies,unique_inn,unique_ogrn
u32,u32,u32
100,100,100


In [ ]:
sample_companies.select([
    pl.col("risk_revenue_drop_gt_30").sum().alias("revenue_drop_cases"),
    pl.col("risk_negative_profit").sum().alias("negative_profit_cases"),
    pl.col("risk_assets_drop_gt_25").sum().alias("assets_drop_cases"),
    pl.col("risk_negative_equity").sum().alias("negative_equity_cases"),
    pl.col("risk_data_quality_issue").sum().alias("data_quality_cases"),
])

revenue_drop_cases,negative_profit_cases,assets_drop_cases,negative_equity_cases,data_quality_cases
u32,u32,u32,u32,u32
32,30,30,18,15


## 9.8. Готовим связанные отчёты только для выбранных компаний

In [ ]:
sample_inns = sample_companies["inn"].to_list()

sample_reports = (
    reports_2y
    .filter(pl.col("inn").is_in(sample_inns))
    .sort(["inn", "year"])
)

print(sample_reports.shape)

sample_reports.select([
    "inn",
    "year",
    "revenue",
    "net_profit",
    "assets",
    "equity",
    "filed",
    "imputed",
    "outlier",
]).head()

(285, 22)


inn,year,revenue,net_profit,assets,equity,filed,imputed,outlier
str,i32,f64,f64,f64,f64,f64,f64,f64
"""1326253200""",2023,5831.0,35.0,142811.0,21031.0,1.0,0.0,0.0
"""1326253200""",2024,0.0,-8809.0,120759.0,12222.0,1.0,0.0,0.0
"""2330033470""",2022,1250.0,-11507.0,51589.0,-9840.0,0.0,1.0,0.0
"""2330033470""",2023,1220.0,-5002.0,45334.0,-4992.0,1.0,0.0,0.0
"""2330033470""",2024,1300.0,-2525.0,31090.0,-3230.0,1.0,0.0,0.0


In [ ]:
sample_reports.group_by("inn").agg(
    pl.col("year").n_unique().alias("years_count")
).group_by("years_count").agg(
    pl.len().alias("companies_count")
).sort("years_count")

years_count,companies_count
u32,u32
2,15
3,85


## 9.9. Результат раздела 9

После этого у нас есть два ключевых датафрейма:

sample_companies — итоговая PoC-выборка компаний с рассчитанными признаками
sample_reports   — отчётность выбранных компаний за 2023 и 2024 годы

# 10. Выгрузить и скачать



In [ ]:
poc_company = (
    sample_companies
    .select([c for c in COMPANY_COLUMNS + ["company_label", "sample_bucket"] if c in sample_companies.columns])
    .unique(subset=["inn"], keep="first")
    .sort("inn")
)

print(poc_company.shape)
poc_company.head()

(100, 11)


inn,ogrn,region,region_taxcode,okved,okved_section,creation_date,dissolution_date,age,company_label,sample_bucket
str,str,str,str,str,str,date,date,f64,str,str
"""1326253200""","""1191326001680""","""moscow city""","""7700""","""47.11""","""G""",2019-03-07,null,5.0,"""Компания ИНН 1326253200""","""negative_profit"""
"""2330033470""","""1062330008685""","""moscow city""","""7700""","""10.72""","""C""",2006-09-28,null,18.0,"""Компания ИНН 2330033470""","""negative_profit"""
"""5001124939""","""1195081013610""","""moscow reg.""","""5000""","""46.74""","""G""",2019-03-06,null,5.0,"""Компания ИНН 5001124939""","""revenue_drop_gt_30"""
"""5010060470""","""1225000120124""","""moscow reg.""","""5000""","""62.01""","""J""",2022-10-24,null,2.0,"""Компания ИНН 5010060470""","""negative_profit"""
"""5018124768""","""1085018000979""","""moscow reg.""","""5000""","""25.62""","""C""",2008-02-04,null,16.0,"""Компания ИНН 5018124768""","""revenue_drop_gt_30"""


In [ ]:
poc_financial_report = (
    sample_reports
    .select([c for c in REPORT_COLUMNS if c in sample_reports.columns])
    .sort(["inn", "year"])
)

print(poc_financial_report.shape)
poc_financial_report.head()

(285, 22)


inn,ogrn,year,region,region_taxcode,okved,okved_section,creation_date,dissolution_date,age,eligible,filed,imputed,outlier,revenue,net_profit,assets,equity,longterm_liab,shortterm_liab,payables,cash
str,str,i32,str,str,str,str,date,date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""1326253200""","""1191326001680""",2023,"""moscow city""","""7700""","""47.11""","""G""",2019-03-07,null,4.0,1.0,1.0,0.0,0.0,5831.0,35.0,142811.0,21031.0,null,121780.0,54052.0,36.0
"""1326253200""","""1191326001680""",2024,"""moscow city""","""7700""","""47.11""","""G""",2019-03-07,null,5.0,1.0,1.0,0.0,0.0,0.0,-8809.0,120759.0,12222.0,null,108537.0,12293.0,100.0
"""2330033470""","""1062330008685""",2022,"""moscow city""","""7700""","""10.72""","""C""",2006-09-28,null,16.0,1.0,0.0,1.0,0.0,1250.0,-11507.0,51589.0,-9840.0,null,61429.0,null,null
"""2330033470""","""1062330008685""",2023,"""moscow city""","""7700""","""10.72""","""C""",2006-09-28,null,17.0,1.0,1.0,0.0,0.0,1220.0,-5002.0,45334.0,-4992.0,null,50326.0,0.0,0.0
"""2330033470""","""1062330008685""",2024,"""moscow city""","""7700""","""10.72""","""C""",2006-09-28,null,18.0,1.0,1.0,0.0,0.0,1300.0,-2525.0,31090.0,-3230.0,null,34320.0,null,368.0


In [ ]:
poc_company_features = (
    sample_companies
    .select([
        c for c in ["inn", "ogrn", "sample_bucket"] + DERIVED_FEATURE_COLUMNS + RISK_COLUMNS + ["risk_count"]
        if c in sample_companies.columns
    ])
    .unique(subset=["inn"], keep="first")
    .sort("inn")
)

print(poc_company_features.shape)
poc_company_features.head()

(100, 24)


inn,ogrn,sample_bucket,revenue_2023,revenue_2024,revenue_drop_2024_pct,net_profit_2023,net_profit_2024,assets_2023,assets_2024,assets_drop_2024_pct,equity_2023,equity_2024,filed_2023,filed_2024,imputed_2023,imputed_2024,outlier_2023,outlier_2024,risk_revenue_drop_gt_30,risk_negative_profit,risk_assets_drop_gt_25,risk_negative_equity,risk_data_quality_issue
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool
"""1326253200""","""1191326001680""","""negative_profit""",5831.0,0.0,100.0,35.0,-8809.0,142811.0,120759.0,15.441388,21031.0,12222.0,1.0,1.0,0.0,0.0,0.0,0.0,true,true,false,false,false
"""2330033470""","""1062330008685""","""negative_profit""",1220.0,1300.0,-6.557377,-5002.0,-2525.0,45334.0,31090.0,31.420126,-4992.0,-3230.0,1.0,1.0,0.0,0.0,0.0,0.0,false,true,true,true,false
"""5001124939""","""1195081013610""","""revenue_drop_gt_30""",15438.0,7539.0,51.165954,227.0,139.0,10597.0,11087.0,-4.62395,241.0,304.0,1.0,1.0,0.0,0.0,0.0,0.0,true,false,false,false,false
"""5010060470""","""1225000120124""","""negative_profit""",null,null,null,-511.0,-178.0,229.0,58.0,74.672489,205.0,27.0,1.0,1.0,0.0,0.0,0.0,0.0,false,true,true,false,false
"""5018124768""","""1085018000979""","""revenue_drop_gt_30""",10282.0,5373.0,47.74363,2784.0,1270.0,19568.0,6596.0,66.291905,18096.0,6542.0,1.0,1.0,0.0,0.0,0.0,0.0,true,false,true,false,false


In [ ]:
POC_COMPANY_PATH = DATA_DIR / "poc_company.csv"
POC_FINANCIAL_REPORT_PATH = DATA_DIR / "poc_financial_report.csv"
POC_COMPANY_FEATURES_PATH = DATA_DIR / "poc_company_features.csv"

poc_company.write_csv(POC_COMPANY_PATH)
poc_financial_report.write_csv(POC_FINANCIAL_REPORT_PATH)
poc_company_features.write_csv(POC_COMPANY_FEATURES_PATH)

print("Saved files:")
print(POC_COMPANY_PATH)
print(POC_FINANCIAL_REPORT_PATH)
print(POC_COMPANY_FEATURES_PATH)

Saved files:
data/poc_company.csv
data/poc_financial_report.csv
data/poc_company_features.csv


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

DATA_DIR = Path("data")
ZIP_PATH = "poc_dataset.zip"

csv_files = [
    DATA_DIR / "poc_company.csv",
    DATA_DIR / "poc_financial_report.csv",
    DATA_DIR / "poc_company_features.csv",
]

with ZipFile(ZIP_PATH, "w") as zipf:
    for csv_file in csv_files:
        zipf.write(csv_file, arcname=csv_file.name)

files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>